In [168]:
import torch
from torch import nn
from torch.nn import functional as F

net  = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
X = torch.rand(2,20)
net(X)

tensor([[ 0.1176,  0.2165, -0.1938, -0.1059,  0.1657,  0.1365, -0.0885,  0.0453,
         -0.0747,  0.2873],
        [ 0.2427,  0.1154, -0.1923, -0.2460,  0.0292,  0.1894, -0.0794,  0.1271,
         -0.1610,  0.2935]], grad_fn=<AddmmBackward0>)

In [172]:
class MLP(nn.Module):
    # 用模型参数声明层这里我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params
        super().__init__()
        self.hidden = nn.Linear(20,256) # 隐藏层
        self.out = nn.Linear(256,10) # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self,X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义
        return self.out(F.relu(self.hidden(X)))

net = MLP()
net(X)

tensor([[ 0.2080, -0.2955,  0.0673,  0.0611, -0.4988, -0.2232,  0.0815,  0.0747,
         -0.2143, -0.3279],
        [ 0.1394, -0.2708, -0.0861,  0.0704, -0.4334, -0.1682,  0.0515, -0.0014,
         -0.2385, -0.1373]], grad_fn=<AddmmBackward0>)

In [174]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for idx,module in enumerate(args):
            # 这里，module是Module子类的一个实例我们把它保存在Module类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self,X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

net = MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(X)

tensor([[-0.2527, -0.2759, -0.1346, -0.1835, -0.2419, -0.0327, -0.0627, -0.0447,
          0.0697,  0.3325],
        [-0.2647, -0.3150, -0.1385, -0.0284, -0.3608,  0.0567, -0.0203, -0.0255,
          0.0299,  0.2073]], grad_fn=<AddmmBackward0>)

In [177]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数，因此在其训练期间保持不变
        self.rand_weight = torch.rand((20,20),requires_grad=False)
        self.linear = nn.Linear(20,20)

    def forward(self,X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X,self.rand_weight)+1)
        # 复用全连接层这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
net(X)

tensor(-0.0824, grad_fn=<SumBackward0>)

In [179]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())
        self.linear = nn.Linear(32,16)

    def forward(self,X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(),nn.Linear(16,20),FixedHiddenMLP())
chimera(X)

tensor(0.6765, grad_fn=<SumBackward0>)